In [ ]:
!pip install genaibook

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.4/282.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=78342d09e214fc0be934abad82d19bdbb43d7afe5413cf851d848cb1dc040630
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Fine-Tuning **SmolLM-135M** model for Generating Sports News:

 - Utilized **AG News Dataset** for a generative task using the **Sports** news available in it.
    

In [ ]:
# Importing Libraries:

import numpy as np
import pandas as pd
import torch

import datasets
from datasets import load_dataset

import transformers

# Preprocessing:
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling

# Training:
from transformers import TrainingArguments, Trainer

# Post Training Analysis:
from transformers import pipeline
import evaluate
import re

## Loading **AG News** Dataset:


In [ ]:
news_dataset = load_dataset("ag_news")
news_dataset

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [ ]:
# Training Dataset and features:
news_train_dataset = news_dataset["train"]
print(news_train_dataset.features)


{'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'], id=None)}


In [ ]:
# Sports News:
news_train_dataset[1300]

{'text': "Phelps's chase of Spitz mark? It's history This was the event Michael Phelps didn't really need to compete in if his goal was to win eight golds. He probably would have had a better chance somewhere else.",
 'label': 1}

In [ ]:
# Defining news id to label map:
news_id_to_label_map = { 0:"World", 1:"Sports", 2:"Business", 3:"Sci/Tech" }

In [ ]:
# Filtering the sports news dataset using label_id:
sports_datasets = news_dataset.filter(lambda example: example["label"] == 1)
sports_datasets = sports_datasets.remove_columns("label")

Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7600 [00:00<?, ? examples/s]

## Preprocessing:

### Loading the tokenizer for SmolLM-135M:


In [ ]:
# Loading tokenizer for SmolLM-135M:
model_name = "HuggingFaceTB/SmolLM2-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)


tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

In [ ]:
# We need to specify as SmolLM's tokenizer doesn't include the padding token:
tokenizer.pad_token = ( tokenizer.eos_token )

In [ ]:
# Define Tokenizer wrapper function:
def tokenizer_wrapper(batch):
    return tokenizer(batch["text"], truncation=True)


### Tokenizing the Sports News Dataset:

In [ ]:
# Note: We require input_ids and attention_mask
# Since we can straight up work with token ids:

tokenized_sports_news_datasets = sports_datasets.map(
                                    tokenizer_wrapper,
                                    batched = True,
                                    remove_columns = ["text"],
                                )


Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1900 [00:00<?, ? examples/s]

In [ ]:
tokenized_sports_news_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1900
    })
})

In [ ]:
# Showing example tokenization:
ip_index = 69
example_tokenized = tokenized_sports_news_datasets['train'][ip_index]
example_tokenized_input_ids = list(example_tokenized['input_ids'])
example_tokenized_attention_mask = list(example_tokenized['attention_mask'])

# Showing example tokenization:
print(f"tokenized_ids:\n{example_tokenized_input_ids}")
print(f"\n\nattention_mask:\n{example_tokenized_attention_mask}")



tokenized_ids:
[44745, 917, 8992, 18968, 370, 216, 35, 29, 33, 288, 48750, 31732, 534, 365, 3872, 25, 6594, 731, 41710, 8581, 12713, 3917, 582, 1658, 281, 2976, 7954, 616, 327, 650, 808, 4726, 281, 3920, 253, 3531, 284, 4573, 4653, 10463, 2994, 253, 1296, 29, 10521, 24190, 282, 260, 11554, 18968, 370, 351, 253, 216, 35, 29, 33, 9970, 10528, 30]


attention_mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Training the Model for Fine-Tuning:


In [ ]:
# Identifying device to train on GPU:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
# The parameter `mlm` ==> masked language modeling
# Since we are doing Causal Learning, we set:
# mlm = False

# Initialising the data collator:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
# Reviewing shape of the tokenized texts inputs:
samples = [tokenized_sports_news_datasets["train"][i] for i in range(5)]

for sample in samples:
    print(f"input_ids shape: {len(sample['input_ids'])}")

input_ids shape: 103
input_ids shape: 81
input_ids shape: 60
input_ids shape: 65
input_ids shape: 51


In [ ]:
# Reviewing shape of the tokenized samples post using data collator:
data_collator_samples_output = data_collator(samples)
for key in data_collator_samples_output:
    print(f"{key} shape: {data_collator_samples_output[key].shape}")

input_ids shape: torch.Size([5, 103])
attention_mask shape: torch.Size([5, 103])
labels shape: torch.Size([5, 103])


### Loading the SmolLM-135M model:

In [ ]:
# Loading the model (SmolLM-135M) for causal learning:
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

### Setting up Training Arguments and Initialising the Trainer:

In [ ]:
# Shuffling dataset to pick 20000 examples to Train/Fine-Tune over:
shuffled_dataset = tokenized_sports_news_datasets["train"].shuffle(seed = 69)
training_subset_data = shuffled_dataset.select(range(20000))
eval_subset_data = tokenized_sports_news_datasets["test"].select(range(1600))

In [ ]:
# Setting the Training Arguments:
# batch_size = 3

# training_args = TrainingArguments(
#     "sports-news-generator",
#     push_to_hub = False,
#     per_device_train_batch_size = batch_size,
#     weight_decay = 0.1,
#     lr_scheduler_type = "cosine",
#     learning_rate = 5e-4,
#     num_train_epochs = 2,
#     eval_strategy = "steps",
#     eval_steps = 200,
#     logging_steps = 200,
#     gradient_accumulation_steps=3,
#     warmup_steps=500)


batch_size = 8

training_args = TrainingArguments(
    "sports-news-generator",
    push_to_hub=False,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    num_train_epochs=3,  # Increased epochs for better fine-tuning
    evaluation_strategy="steps",
    eval_steps=200,
    logging_steps=200,
    save_strategy="steps",  # Save model checkpoints every eval_steps
    save_steps=200,
    save_total_limit=2,  # Limit to 2 checkpoints to save space
    warmup_steps=int(0.03 * (len(training_subset_data) // batch_size)),  # 3% of total steps
    gradient_accumulation_steps=2,  # Simulate larger batch size
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# Initialize the Trainer:
trainer = Trainer(
    model = model,
    tokenizer = tokenizer,
    args = training_args,
    data_collator = data_collator,
    train_dataset = training_subset_data,
    eval_dataset = eval_subset_data,
)

<ipython-input-24-c7b453824b5f>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Start Training/Fine-Tuning the model:

In [ ]:
# Train the model:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ashishmeshram159 (ashishmeshram159-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Step,Training Loss,Validation Loss
200,3.116800,3.345124
400,3.241900,3.240290
600,3.133800,3.182369
800,3.064600,3.088837
1000,3.019800,3.007563
1200,2.947600,2.989421
1400,2.263200,3.093879
1600,2.063500,3.038800
1800,2.082100,2.973291
2000,2.080300,2.918189


TrainOutput(global_step=5000, training_loss=1.5445147239685058, metrics={'train_runtime': 7236.7493, 'train_samples_per_second': 11.055, 'train_steps_per_second': 0.691, 'total_flos': 6.214749607329792e+16, 'train_loss': 1.5445147239685058, 'epoch': 4.0})

In [ ]:
print(12)

12


### Saving the Fine-Tuned model:

In [ ]:
# Saving the Fine-Tuned model in './Sports_News_Generation_model' directory:
trainer.save_model("/content/drive/MyDrive/SportsNewsGeneratorModel/Sports_News_Generation_model/sports_news_gen_model_bs8_ep4_FT_V4_1.7B")


## Post Training/Fine-Tuning Analysis:

In [ ]:
# Initialising pipeline for inferences:
pipe = pipeline(
    "text-generation",
    model="/content/drive/MyDrive/SportsNewsGeneratorModel/Sports_News_Generation_model/sports_news_gen_model_bs8_ep4_FT_V4_1.7B",
    device=device,
)



# Test generation:
print(
    pipe("LA Lakers", do_sample=True, temperature=0.8, max_new_tokens=30)[0][
        "generated_text"
    ]
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


LA Lakers 115, Kings 99 When the Lakers were dismantled last summer, the Sacramento Kings thought their biggest rivalry was finished. Turns out


In [ ]:
# Example generation:
input_prompt_example_1 = "Micheal Phelps wins"
generated_example_1 = pipe(input_prompt_example_1, do_sample=True, temperature=0.7, max_new_tokens=30)[0][
                        "generated_text"
                    ]

input_prompt_example_2 = "Sacramento Kings player"
generated_example_2 = pipe(input_prompt_example_2, do_sample=True, temperature=0.7, max_new_tokens=30)[0][
                        "generated_text"
                    ]


print(f"input_prompt_example_1:\n{input_prompt_example_1}\n\ngenerated_example_1:\n{generated_example_1} ")
print(f"input_prompt_example_2:\n{input_prompt_example_2}\n\ngenerated_example_2:\n{generated_example_2} ")

input_prompt_example_1:
Micheal Phelps wins

generated_example_1:
Micheal Phelps wins duel with Crocker ATHENS, Greece -- The 10-race, 10-event Olympic swimming competition came to a head 
input_prompt_example_2:
Sacramento Kings player

generated_example_2:
Sacramento Kings player arrested on gun charge (AFP) AFP - Sacramento Kings basketball player Mike D'Antoni was arrested on suspicion of gunning down his teammate 


In [ ]:
# Dataset for post training analysis:
# Using 300 samples from the tokenized_sports_news_datasets for post training analysis:
test_subset = tokenized_sports_news_datasets["test"].select(range(1600, 1900)).shuffle(seed = 71)
test_subset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 300
})

In [ ]:
# Function to decode input_ids to text
def decode_input_ids(input_ids):
    return tokenizer.decode(input_ids, skip_special_tokens=True)

# Function to split the text into:
# - prompt (first line of the text) and
# - the rest as reference

def get_first_sentence(text):
    # sentences = re.split(r'(?<=[.!?])\s+', text)
    return " ".join(text.split()[:10]), text



In [ ]:
# Generating prompt-reference dataset:
prompt_reference_data = []

for elem in test_subset:
    text = decode_input_ids(elem["input_ids"])
    prompt, reference = get_first_sentence(text)
    prompt_reference_data.append({"prompt": prompt, "reference": reference})


In [ ]:
# Testing 5 samples:
for i in range(5):
    print(f"Sample {i + 1}:")
    print(f"Prompt: {prompt_reference_data[i]['prompt']}")
    print(f"Reference: {prompt_reference_data[i]['reference']}\n")


Sample 1:
Prompt: Bengals' Palmer Questionable for Sunday CINCINNATI (Sports Network) - Cincinnati
Reference: Bengals' Palmer Questionable for Sunday  CINCINNATI (Sports Network) - Cincinnati Bengals  quarterback Carson Palmer is questionable for Sunday's game  against Buffalo after an MRI exam Monday revealed no serious  damage to his left knee.

Sample 2:
Prompt: Expectations too lofty for unlucky Willingham Three seasons after hiring
Reference: Expectations too lofty for unlucky Willingham Three seasons after hiring Tyrone Willingham as head coach of the football program, the powers that be in South Bend, Ind., fired the 28-year coaching veteran Tuesday, one month prior to the Fighting Irish #39;s scheduled matchup with UCLA in the Insight Bowl 

Sample 3:
Prompt: Red Sox Formula Is a Model for Success As the
Reference: Red Sox Formula Is a Model for Success As the shuffling of players intensifies this off-season, some of the Boston Red Sox' pictures will come down. The champions wi

In [ ]:
# Declaring pipe for text generation:
pipe = pipeline("text-generation", model=model_name, device=device)


Device set to use cuda


In [ ]:
# Prompt Texts:
prompts_texts = [data["prompt"] for data in prompt_reference_data]


In [ ]:
# Generating completions:
# generated_texts = [pipe(data["prompt"], do_sample=True, temperature=0.7, max_new_tokens=42)[0]["generated_text"] for data in prompt_reference_data[:5]]
generated_texts_list = pipe(prompts_texts, do_sample=True, temperature=1, max_new_tokens=42)
# generated_texts_list = generated_texts_list[1:]

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for o

In [ ]:
generated_texts_list[:5]

[[{'generated_text': 'Bengals\' Palmer Questionable for Sunday CINCINNATI (Sports Network) - Cincinnati defensive coordinator Ray Horton says quarterback Jay Cutler has a back injury and could be questionable for Sunday\'s game.\n\nThe team believes he\'ll play.\n\n"We don\'t have any reason'}],
 [{'generated_text': 'Expectations too lofty for unlucky Willingham Three seasons after hiring Mike Shanahan as his offensive coordinator, Dan "Cleveland Browns are a football team again" Reeves decided to get something more out of the offense than the play-calling he inherited from Shanahan'}],
 [{'generated_text': 'Red Sox Formula Is a Model for Success As the Cubs Sink\n\n\nThe new\xa0Red Sox\xa0traded their most promising and marketable player at each position to the\xa0Yankees\xa0and\xa0Phillies\xa0for a'}],
 [{'generated_text': 'George sits for first time in career Irving, TX (Sports Network) - With the Red Sox looking to make the 2013 playoffs as a wild card for the first time since the 

In [ ]:
generated_texts = [generated_text[0]["generated_text"] for generated_text in generated_texts_list]


### Calculating ROUGE and BLEU Scores:

In [ ]:
# Loading BLEU and ROUGE scores from evaluate:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

In [ ]:
# Calculating BLEU scores and ROUGE scores for all 300 samples:
bleu_scores = [bleu.compute(predictions=[generated], references=[data["reference"]]) for data, generated in zip(prompt_reference_data, generated_texts)]
rouge_scores = [rouge.compute(predictions=[generated], references=[data["reference"]]) for data, generated in zip(prompt_reference_data, generated_texts)]


In [ ]:
# Printing BLEU and ROUGE scores for first 5 samples:
print(f"BLEU Scores for the first 5 samples: {bleu_scores[:5]}\n")
print(f"ROUGE Scores for the first 5 samples: {rouge_scores[:5]}")


BLEU Scores for the first 5 samples: [{'bleu': 0.31038939679188, 'precisions': [0.4090909090909091, 0.32558139534883723, 0.2857142857142857, 0.24390243902439024], 'brevity_penalty': 1.0, 'length_ratio': 1.1891891891891893, 'translation_length': 44, 'reference_length': 37}, {'bleu': 0.16533535426671866, 'precisions': [0.38636363636363635, 0.23255813953488372, 0.19047619047619047, 0.17073170731707318], 'brevity_penalty': 0.7111235528636867, 'length_ratio': 0.7457627118644068, 'translation_length': 44, 'reference_length': 59}, {'bleu': 0.2740198653930223, 'precisions': [0.42424242424242425, 0.28125, 0.25806451612903225, 0.23333333333333334], 'brevity_penalty': 0.9411939401248326, 'length_ratio': 0.9428571428571428, 'translation_length': 33, 'reference_length': 35}, {'bleu': 0.3435143170579934, 'precisions': [0.4782608695652174, 0.4, 0.3409090909090909, 0.3023255813953488], 'brevity_penalty': 0.9167169520254864, 'length_ratio': 0.92, 'translation_length': 46, 'reference_length': 50}, {'ble

### For Overall BLEU and ROUGE scores:

In [ ]:
prompts_texts_bleu = [[elem['prompt']] for elem in prompt_reference_data]
prompts_texts_bleu

[["Bengals' Palmer Questionable for Sunday CINCINNATI (Sports Network) - Cincinnati"],
 ['Expectations too lofty for unlucky Willingham Three seasons after hiring'],
 ['Red Sox Formula Is a Model for Success As the'],
 ['George sits for first time in career Irving, TX (Sports'],
 ['Poll cost us victory - Cech Referee Graham Poll came'],
 ['Oxford 18 Cambridge 11 Replacement winger Ross Lavery scored the'],
 ['Barca wins again Barcelona has moved 12 points clear at'],
 ['The Newest Hope ; Marriage of Necessity Just Might Work'],
 ['Football Association charges Bolton striker over spitting incident Bolton striker'],
 ['ITA: Juventus blows 2-goal lead againt Inter Milan Serie A'],
 ['Milan Mandaric statement quot;Over the past two-and-a-half years the football'],
 ['Sherman remains confident after Packers #39; flop in Philly They'],
 ['Utah Hires Whittingham to Replace Meyer (AP) AP - Utah'],
 ['Memphis indefinitely suspends Sean Banks Memphis forward Sean Banks was'],
 ['Delaney keen to 

In [ ]:
# Get reference texts for the generated output:
reference_texts = [[elem['reference']] for elem in prompt_reference_data]
reference_texts[:5]


[["Bengals' Palmer Questionable for Sunday  CINCINNATI (Sports Network) - Cincinnati Bengals  quarterback Carson Palmer is questionable for Sunday's game  against Buffalo after an MRI exam Monday revealed no serious  damage to his left knee."],
 ['Expectations too lofty for unlucky Willingham Three seasons after hiring Tyrone Willingham as head coach of the football program, the powers that be in South Bend, Ind., fired the 28-year coaching veteran Tuesday, one month prior to the Fighting Irish #39;s scheduled matchup with UCLA in the Insight Bowl '],
 ["Red Sox Formula Is a Model for Success As the shuffling of players intensifies this off-season, some of the Boston Red Sox' pictures will come down. The champions will have to change."],
 ['George sits for first time in career Irving, TX (Sports Network) - Dallas Cowboys running back Eddie George was inactive for Sunday #39;s game against New Orleans as a healthy scratch and missed a game for the first time in his NFL career.'],
 ['Pol

In [ ]:
generated_texts[:5]

['Bengals\' Palmer Questionable for Sunday CINCINNATI (Sports Network) - Cincinnati defensive coordinator Ray Horton says quarterback Jay Cutler has a back injury and could be questionable for Sunday\'s game.\n\nThe team believes he\'ll play.\n\n"We don\'t have any reason',
 'Expectations too lofty for unlucky Willingham Three seasons after hiring Mike Shanahan as his offensive coordinator, Dan "Cleveland Browns are a football team again" Reeves decided to get something more out of the offense than the play-calling he inherited from Shanahan',
 'Red Sox Formula Is a Model for Success As the Cubs Sink\n\n\nThe new\xa0Red Sox\xa0traded their most promising and marketable player at each position to the\xa0Yankees\xa0and\xa0Phillies\xa0for a',
 'George sits for first time in career Irving, TX (Sports Network) - With the Red Sox looking to make the 2013 playoffs as a wild card for the first time since the year of the "Sox or Sox?" question in ',
 "Poll cost us victory - Cech Referee Graham 

In [ ]:
# Overall BLEU Score:
overall_bleu_score = bleu.compute(predictions=generated_texts, references=reference_texts)
print(f"Overall BLEU Score: {overall_bleu_score['bleu']:.4f}")


Overall BLEU Score: 0.2731


In [ ]:
# ROUGE scores:
rouge_1_f1 = [score['rouge1'] for score in rouge_scores]
rouge_2_f1 = [score['rouge2'] for score in rouge_scores]
rouge_l_f1 = [score['rougeL'] for score in rouge_scores]

# Calculate average ROUGE F1 scores:
average_rouge_1_f1 = np.mean(rouge_1_f1)
average_rouge_2_f1 = np.mean(rouge_2_f1)
average_rouge_l_f1 = np.mean(rouge_l_f1)

# Print the average ROUGE scores:
print(f"Average ROUGE-1 F1: {average_rouge_1_f1}")
print(f"Average ROUGE-2 F1: {average_rouge_2_f1}")
print(f"Average ROUGE-L F1: {average_rouge_l_f1}")

Average ROUGE-1 F1: 0.39114890390666146
Average ROUGE-2 F1: 0.2712963378872219
Average ROUGE-L F1: 0.3598925484537796


In [ ]:
import torch
torch.cuda.empty_cache()
